# FSPD Multi-Task Classification Training
## XLM-RoBERTa for Food System Policy Classification

This notebook trains a transformer model to classify food system policies across 6 tasks:
- Segment (11 classes)
- Lever (113 classes)
- Type (9 classes)  
- Inclusion (binary)
- Environmental target (binary)
- Diet health (binary)

### Setup Instructions for Kaggle:
1. Upload `FSPD.csv` as a Kaggle dataset
2. Enable GPU: Settings → Accelerator → GPU T4 x2
3. Add the FSPD dataset to this notebook
4. Run all cells

In [ ]:
# Install dependencies
!pip install -q transformers datasets

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import json
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
# Configuration
CONFIG = {
    'base_model': 'xlm-roberta-base',
    'max_length': 512,
    'batch_size': 16,  # Adjust based on GPU memory
    'learning_rate': 2e-5,
    'num_epochs': 10,
    'warmup_steps': 500,
    'weight_decay': 0.01,
    'random_seed': 42,
    'use_class_weights': True,
    'patience': 3,
    
    # Task weights
    'segment_weight': 1.0,
    'lever_weight': 1.0,
    'type_weight': 1.0,
    'inclusion_weight': 0.5,
    'envitarget_weight': 0.5,
    'diethealth_weight': 0.5,
    
    # Data - CHANGE THIS TO YOUR KAGGLE DATASET PATH
    'data_path': '/kaggle/input/fspd/FSPD.csv',  # Update this path!
    'output_dir': '/kaggle/working/models',
}

# Set seed
torch.manual_seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])

os.makedirs(CONFIG['output_dir'], exist_ok=True)

## Dataset Class

In [ ]:
class FSPDDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512, 
                 segment_encoder=None, lever_encoder=None, type_encoder=None, is_training=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        self.segment_encoder = segment_encoder or LabelEncoder()
        self.lever_encoder = lever_encoder or LabelEncoder()
        self.type_encoder = type_encoder or LabelEncoder()
        
        if is_training and segment_encoder is None:
            self.segment_encoder.fit(self.dataframe['segment'])
            self.lever_encoder.fit(self.dataframe['lever'])
            self.type_encoder.fit(self.dataframe['itype'])
        
        self.dataframe['segment_encoded'] = self.segment_encoder.transform(self.dataframe['segment'])
        self.dataframe['lever_encoded'] = self.lever_encoder.transform(self.dataframe['lever'])
        self.dataframe['type_encoded'] = self.type_encoder.transform(self.dataframe['itype'])
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        text = str(row['policydecision_details'])
        
        encoding = self.tokenizer(text, max_length=self.max_length, padding='max_length',
                                 truncation=True, return_tensors='pt')
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'segment_label': torch.tensor(row['segment_encoded'], dtype=torch.long),
            'lever_label': torch.tensor(row['lever_encoded'], dtype=torch.long),
            'type_label': torch.tensor(row['type_encoded'], dtype=torch.long),
            'inclusion_label': torch.tensor(row['inclusion'], dtype=torch.float),
            'envitarget_label': torch.tensor(row['envitarget'], dtype=torch.float),
            'diethealth_label': torch.tensor(row['diethealth'], dtype=torch.float),
        }
    
    def get_class_weights(self, task='segment'):
        if task == 'segment':
            labels = self.dataframe['segment_encoded']
            n_classes = len(self.segment_encoder.classes_)
        elif task == 'lever':
            labels = self.dataframe['lever_encoded']
            n_classes = len(self.lever_encoder.classes_)
        elif task == 'type':
            labels = self.dataframe['type_encoded']
            n_classes = len(self.type_encoder.classes_)
        
        class_counts = np.bincount(labels, minlength=n_classes)
        class_weights = 1.0 / (class_counts + 1)
        class_weights = class_weights / class_weights.sum() * n_classes
        return torch.FloatTensor(class_weights)

## Model Architecture

In [ ]:
class FSPDMultiTaskModel(nn.Module):
    def __init__(self, model_name, num_segments, num_levers, num_types, dropout_rate=0.1):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name)
        self.transformer = AutoModel.from_pretrained(model_name, config=config)
        hidden_size = config.hidden_size
        
        self.dropout = nn.Dropout(dropout_rate)
        self.segment_classifier = nn.Linear(hidden_size, num_segments)
        self.lever_classifier = nn.Linear(hidden_size, num_levers)
        self.type_classifier = nn.Linear(hidden_size, num_types)
        self.inclusion_classifier = nn.Linear(hidden_size, 1)
        self.envitarget_classifier = nn.Linear(hidden_size, 1)
        self.diethealth_classifier = nn.Linear(hidden_size, 1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        
        return {
            'segment_logits': self.segment_classifier(pooled_output),
            'lever_logits': self.lever_classifier(pooled_output),
            'type_logits': self.type_classifier(pooled_output),
            'inclusion_logits': self.inclusion_classifier(pooled_output).squeeze(-1),
            'envitarget_logits': self.envitarget_classifier(pooled_output).squeeze(-1),
            'diethealth_logits': self.diethealth_classifier(pooled_output).squeeze(-1),
        }

class MultiTaskLoss(nn.Module):
    def __init__(self, segment_weight=1.0, lever_weight=1.0, type_weight=1.0,
                 inclusion_weight=0.5, envitarget_weight=0.5, diethealth_weight=0.5,
                 segment_class_weights=None, lever_class_weights=None, type_class_weights=None):
        super().__init__()
        self.segment_weight = segment_weight
        self.lever_weight = lever_weight
        self.type_weight = type_weight
        self.inclusion_weight = inclusion_weight
        self.envitarget_weight = envitarget_weight
        self.diethealth_weight = diethealth_weight
        
        self.segment_loss_fn = nn.CrossEntropyLoss(weight=segment_class_weights)
        self.lever_loss_fn = nn.CrossEntropyLoss(weight=lever_class_weights)
        self.type_loss_fn = nn.CrossEntropyLoss(weight=type_class_weights)
        self.binary_loss_fn = nn.BCEWithLogitsLoss()
    
    def forward(self, outputs, labels):
        segment_loss = self.segment_loss_fn(outputs['segment_logits'], labels['segment_label'])
        lever_loss = self.lever_loss_fn(outputs['lever_logits'], labels['lever_label'])
        type_loss = self.type_loss_fn(outputs['type_logits'], labels['type_label'])
        inclusion_loss = self.binary_loss_fn(outputs['inclusion_logits'], labels['inclusion_label'])
        envitarget_loss = self.binary_loss_fn(outputs['envitarget_logits'], labels['envitarget_label'])
        diethealth_loss = self.binary_loss_fn(outputs['diethealth_logits'], labels['diethealth_label'])
        
        total_loss = (self.segment_weight * segment_loss + self.lever_weight * lever_loss +
                     self.type_weight * type_loss + self.inclusion_weight * inclusion_loss +
                     self.envitarget_weight * envitarget_loss + self.diethealth_weight * diethealth_loss)
        
        return total_loss, {
            'segment': segment_loss.item(), 'lever': lever_loss.item(), 'type': type_loss.item(),
            'inclusion': inclusion_loss.item(), 'envitarget': envitarget_loss.item(), 'diethealth': diethealth_loss.item()
        }

## Load and Prepare Data

In [ ]:
# Load data
print("Loading data...")
df = pd.read_csv(CONFIG['data_path'], low_memory=False)

# Filter and clean
df = df[df['policydecision_details'].notna()].copy()
df = df[df['segment'].notna()]
df = df[df['lever'].notna()]
df = df[df['itype'].notna()]

for col in ['inclusion', 'envitarget', 'diethealth']:
    df[col] = df[col].fillna(0).astype(int)

df = df[df['policydecision_details'].str.len() >= 10]

print(f"Total samples: {len(df)}")
print(f"\nData distribution:")
print(f"  Source databases:\n{df['db'].value_counts()}")
print(f"\n  Segments: {df['segment'].nunique()}")
print(f"  Levers: {df['lever'].nunique()}")
print(f"  Types: {df['itype'].nunique()}")

In [ ]:
# Split data
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=CONFIG['random_seed'], 
                                     stratify=df['segment'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=CONFIG['random_seed'],
                                   stratify=temp_df['segment'])

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# Load tokenizer and create datasets
print(f"Loading tokenizer: {CONFIG['base_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG['base_model'])

train_dataset = FSPDDataset(train_df, tokenizer, CONFIG['max_length'], is_training=True)
val_dataset = FSPDDataset(val_df, tokenizer, CONFIG['max_length'],
                          segment_encoder=train_dataset.segment_encoder,
                          lever_encoder=train_dataset.lever_encoder,
                          type_encoder=train_dataset.type_encoder, is_training=False)
test_dataset = FSPDDataset(test_df, tokenizer, CONFIG['max_length'],
                           segment_encoder=train_dataset.segment_encoder,
                           lever_encoder=train_dataset.lever_encoder,
                           type_encoder=train_dataset.type_encoder, is_training=False)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

print(f"\nDatasets created:")
print(f"  Segments: {len(train_dataset.segment_encoder.classes_)}")
print(f"  Levers: {len(train_dataset.lever_encoder.classes_)}")
print(f"  Types: {len(train_dataset.type_encoder.classes_)}")

## Initialize Model and Training

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = FSPDMultiTaskModel(
    CONFIG['base_model'],
    len(train_dataset.segment_encoder.classes_),
    len(train_dataset.lever_encoder.classes_),
    len(train_dataset.type_encoder.classes_)
)
model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Get class weights
segment_weights = None
lever_weights = None
type_weights = None

if CONFIG['use_class_weights']:
    segment_weights = train_dataset.get_class_weights('segment').to(device)
    lever_weights = train_dataset.get_class_weights('lever').to(device)
    type_weights = train_dataset.get_class_weights('type').to(device)
    print("Using class weights for imbalanced data")

loss_fn = MultiTaskLoss(
    segment_weight=CONFIG['segment_weight'],
    lever_weight=CONFIG['lever_weight'],
    type_weight=CONFIG['type_weight'],
    inclusion_weight=CONFIG['inclusion_weight'],
    envitarget_weight=CONFIG['envitarget_weight'],
    diethealth_weight=CONFIG['diethealth_weight'],
    segment_class_weights=segment_weights,
    lever_class_weights=lever_weights,
    type_class_weights=type_weights
)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'], 
                             weight_decay=CONFIG['weight_decay'])

total_steps = len(train_loader) * CONFIG['num_epochs']
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=CONFIG['warmup_steps'],
                                           num_training_steps=total_steps)

## Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    progress_bar = tqdm(loader, desc="Training")
    
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = {k: v.to(device) for k, v in batch.items() if k.endswith('_label')}
        
        outputs = model(input_ids, attention_mask)
        loss, _ = loss_fn(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})
    
    return total_loss / len(loader)

def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds = {k: [] for k in ['segment', 'lever', 'type', 'inclusion', 'envitarget', 'diethealth']}
    all_labels = {k: [] for k in ['segment', 'lever', 'type', 'inclusion', 'envitarget', 'diethealth']}
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = {k: v.to(device) for k, v in batch.items() if k.endswith('_label')}
            
            outputs = model(input_ids, attention_mask)
            loss, _ = loss_fn(outputs, labels)
            total_loss += loss.item()
            
            # Predictions
            all_preds['segment'].extend(torch.argmax(outputs['segment_logits'], dim=-1).cpu().numpy())
            all_preds['lever'].extend(torch.argmax(outputs['lever_logits'], dim=-1).cpu().numpy())
            all_preds['type'].extend(torch.argmax(outputs['type_logits'], dim=-1).cpu().numpy())
            all_preds['inclusion'].extend((torch.sigmoid(outputs['inclusion_logits']) > 0.5).long().cpu().numpy())
            all_preds['envitarget'].extend((torch.sigmoid(outputs['envitarget_logits']) > 0.5).long().cpu().numpy())
            all_preds['diethealth'].extend((torch.sigmoid(outputs['diethealth_logits']) > 0.5).long().cpu().numpy())
            
            all_labels['segment'].extend(labels['segment_label'].cpu().numpy())
            all_labels['lever'].extend(labels['lever_label'].cpu().numpy())
            all_labels['type'].extend(labels['type_label'].cpu().numpy())
            all_labels['inclusion'].extend(labels['inclusion_label'].cpu().numpy())
            all_labels['envitarget'].extend(labels['envitarget_label'].cpu().numpy())
            all_labels['diethealth'].extend(labels['diethealth_label'].cpu().numpy())
    
    metrics = {}
    for task in all_preds.keys():
        acc = accuracy_score(all_labels[task], all_preds[task])
        avg = 'weighted' if task in ['segment', 'lever', 'type'] else 'binary'
        f1 = f1_score(all_labels[task], all_preds[task], average=avg)
        metrics[f'{task}_acc'] = acc
        metrics[f'{task}_f1'] = f1
    
    return total_loss / len(loader), metrics

In [ ]:
# Training
best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_metrics': []}

print("\nStarting training...\n")
for epoch in range(CONFIG['num_epochs']):
    print(f"Epoch {epoch + 1}/{CONFIG['num_epochs']}")
    
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, device)
    val_loss, val_metrics = evaluate(model, val_loader, loss_fn, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_metrics'].append(val_metrics)
    
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Val Accuracies: " + ", ".join([f"{k}: {v:.3f}" for k, v in val_metrics.items() if 'acc' in k]))
    print(f"  Val F1 Scores: " + ", ".join([f"{k}: {v:.3f}" for k, v in val_metrics.items() if 'f1' in k]))
    
    if val_loss < best_val_loss - 0.001:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_metrics': val_metrics,
            'segment_encoder': train_dataset.segment_encoder,
            'lever_encoder': train_dataset.lever_encoder,
            'type_encoder': train_dataset.type_encoder,
            'config': CONFIG
        }, os.path.join(CONFIG['output_dir'], 'best_model.pt'))
        print("  -> Saved best model")
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f"Early stopping at epoch {epoch + 1}")
            break
    
    print()

## Test Evaluation

In [ ]:
# Load best model and evaluate on test set
checkpoint = torch.load(os.path.join(CONFIG['output_dir'], 'best_model.pt'))
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_metrics = evaluate(model, test_loader, loss_fn, device)

print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)
print(f"Test Loss: {test_loss:.4f}\n")
for k, v in test_metrics.items():
    print(f"{k}: {v:.3f}")

# Save test metrics
with open(os.path.join(CONFIG['output_dir'], 'test_metrics.json'), 'w') as f:
    json.dump(test_metrics, f, indent=2)

print(f"\nModel and metrics saved to {CONFIG['output_dir']}")

## Download Model

After training completes, download `best_model.pt` from the output folder to use for predictions.